11 - Structured Output

This notebook converts the Python tutorial `$11 structured output` into an interactive Jupyter format.

This section sets up the imports and environment configuration needed for the tutorial.

In [ ]:
"""11 - Structured Output======================- Pass a Pydantic model as `response_format` to get typed results- Read it from `result["structured_response"]` — a real object, not a string- No more regex-parsing the model's proseRun:  python 11-structured_output.py"""

This section implements imports and setup.

In [ ]:
# %% Step 1: Imports and setupimport osfrom deepagents import create_deep_agentfrom dotenv import load_dotenvfrom pydantic import BaseModel, Fieldfrom rich import printload_dotenv()MODEL = os.getenv("DEEPAGENTS_MODEL", "ollama:qwen3.5:2b")

This section implements describe the shape you want back.

In [ ]:
# %% Step 2: Describe the shape you want back# Field descriptions are sent to the model as part of the schema, so they act# as per-field instructions. Keep schemas flat and simple — smaller models# handle a few scalar fields far more reliably than deep nesting.class BookReview(BaseModel):    """A structured verdict on a book."""    title: str = Field(description="The title of the book being reviewed.")    rating: int = Field(description="A rating from 1 to 5, where 5 is best.")    one_liner: str = Field(description="A single sentence summarizing the verdict.")

This section implements attach the schema to the agent.

In [ ]:
# %% Step 3: Attach the schema to the agentagent = create_deep_agent(    model=MODEL,    response_format=BookReview,    system_prompt="You are a book critic. Answer using the structured format.",)

This section implements ask a question whose answer fits the schema.

In [ ]:
# %% Step 4: Ask a question whose answer fits the schematask = (    "Review the book 'The Hobbit' by J.R.R. Tolkien in the structured format. "    "Do not use any tools.")# Structured output relies on the model following a tool/JSON schema. Smaller# local models sometimes drift, so fail gracefully rather than crashing.try:    result = agent.invoke({"messages": [{"role": "user", "content": task}]})    review = result.get("structured_response")except Exception as error:  # noqa: BLE001 - tutorial-friendly catch-all    print(f"[red]The model did not produce valid structured output:[/red] {error}")    review = None

This section implements use it like the python object it is.

In [ ]:
# %% Step 5: Use it like the Python object it isif isinstance(review, BookReview):    print("[bold cyan]Typed object returned:[/bold cyan]")    print(f"  type:      [yellow]{type(review).__name__}[/yellow]")    print(f"  title:     {review.title}")    print(f"  rating:    {'★' * review.rating}{'☆' * (5 - review.rating)} ({review.rating}/5)")    print(f"  one_liner: {review.one_liner}")    # Because it is a Pydantic model, you get validation and serialization free.    print("\n[bold cyan]As JSON (ready for an API or database):[/bold cyan]")    print(review.model_dump_json(indent=2))else:    print("[yellow]No structured response — try a larger model.[/yellow]")# %%